# Problèmes traités

## Problème 1 : Couloir

Un couloir peut être imaginé comme une série de cases les unes à côtés des autres.

La longueur du couloir correspond au nombre de cases qui le représentent

L'agent ne perçoit qu'un entier, celui correspondant à la case où il se trouve.
S'il se trouve sur la première case alors il observe 0, il observera 1 s'il se trouve sur la seconde case etc..

Soit n la position actuelle de l'agent.

À chaque pas, l'agent peut effectuer deux actions, aller à gauche (se déplacer en n-1 pour n non nul) ou aller à droite (se déplacer en n+1).
L'épisode se termine lorsque l'agent se trouve en (taille du couloir - 1)


Les récompenses sont distribuées ainsi :
- Si l'agent n'est pas en taille - 1 : -1
- Si l'agent se trouve en taille - 1 : 10

In [ ]:
import gymnasium
from gymnasium import spaces

class LineWorldEnv(gymnasium.Env) :
    def __init__(self, options = None) :
        super(LineWorldEnv, self).__init__()

        # Récupération de la taille du couloir
        self.size = options['n_states']

        # Définition des ensembles
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Discrete(self.size)

        # Initialisation
        self.state = 0

    
    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)

        # Réinitialisation
        self.state = 0

        return self.state, {}

    
    def step(self, action) :

        # On applique l'action
        if action == 0 and self.state > 0 :
            self.state -= 1
        elif action == 1 and self.state < self.size - 1 :
            self.state += 1

        # L'épisode est-il terminé ? Et calcul récompense
        done = self.state == self.size - 1
        reward = 10 if done else -1

        return self.state, reward, done, False, {}


    def render(self) :
        print("État actuel :", self.state)

## Problème 2 : Labyrinthe

L'objectif est d'atteindre la sortie d'un labyrinthe.
Le labyrinthe est représenté par une succession de cases formant un carré dont on peut choisir la taille.

À chaque case est associé un numéro qui représente sa position dans le labyrinthe.
L'agent ne peut perçevoir que le numéro associé à sa position actuelle.

L'agent résoud les labyrinthes un par un (il s'entraîne sur un unique labyrinthe pour en trouver la solution).



À chaque pas, l'agent peut :
- aller en haut
- aller en bas
- aller à gauche
- aller à droite

Mais il n'est pas possible de sortir des limites du labyrinthe (aller en haut en étant tout en haut n'a donc aucun effet)


Le labyrinthe possède de multiples case :
- "F" : une case libre
- "G" : la sortie du labyrinthe
- "S" : le point de départ de l'agent
- "." : la position actuelle de l'agent
- "H" : un trou

  Dans ce labyrinthe, les murs sont remplacés par des trous.

L'épisode prend fin si :
- l'agent tombe dans un trou
- l'agent trouve la sortie

La "taille" du labyrinthe fait référence à la longueur du côté du carré le représentant. Il possède en tout taille^2 cases.

Le point de départ de l'agent est toujours en haut à gauche et la sortie toujours en bas à droite.
Quelque soit le labyrinthe, il existe au moins une solution.

### Labyrinthe 1

La distribution des récompenses est la suivante :
- pour tout déplacement : 0
- trouver la sortie : 1

In [ ]:
class FrozenLakeEnv(gymnasium.Env) :
    def __init__(self, options = None) :
        super(FrozenLakeEnv, self).__init__()
        self.map = options['lake_map']
        self.width = len(self.map[0])
        self.height = len(self.map)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(self.width * self.height)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
    
    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
        return self.state, {}
    
    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.height -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.width -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        self.state = self._coordinateToState(x, y)
        
        result = self.map[y][x]
        done = result in ['H', 'G']
        if result == 'G' :
            reward = 1
        elif result == 'H' :
            reward = 0
        else :
            reward = 0
        return self.state, reward, done, False, {}
    
    
    def _stateToCoordinate(self, state) :
        x = state % self.width
        y = state // self.width
        return x, y
    
    def _coordinateToState(self, x, y) :
        return self.width*y + x
    
    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.height) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.width) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)

### Labyrinthe 2

La distribution des récompenses est la suivante :
- tomber dans un trou : -1
- trouver la sortie : 1
- autre déplacement : -1/nombre_de_cases

In [ ]:
class FrozenLake2Env(gymnasium.Env) :
    def __init__(self, options = None) :
        super(FrozenLake2Env, self).__init__()
        self.map = options['lake_map']
        self.width = len(self.map[0])
        self.height = len(self.map)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(self.width * self.height)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
    
    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
        return self.state, {}
    
    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.height -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.width -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        self.state = self._coordinateToState(x, y)
        
        result = self.map[y][x]
        done = result in ['H', 'G']
        if result == 'G' :
            reward = 1
        elif result == 'H' :
            reward = -1
        else :
            reward = -1/(self.width * self.height)
        return self.state, reward, done, False, {}
    
    
    def _stateToCoordinate(self, state) :
        x = state % self.width
        y = state // self.width
        return x, y
    
    def _coordinateToState(self, x, y) :
        return self.width*y + x
    
    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.height) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.width) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)

### Labyrinthe 3

Même distribution de récompense que labyrinthe 2.

En plus d'observer sa position, l'agent observe les 4 cases adjacentes.

In [ ]:
class FrozenLake3Env(gymnasium.Env) :
    def __init__(self, options = None) :
        super(FrozenLake3Env, self).__init__()
        self.map = options['lake_map']
        self.width = len(self.map[0])
        self.height = len(self.map)
        self.cell_values = {'F' : 0, 'S' : 0, 'H' : 1, 'G' : 2, 'W' : 3}
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.MultiDiscrete([self.width * self.height, 4, 4, 4, 4])
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
        

    
    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break


        left, down, right, up = self._getNeighbors(self.state)
        return np.array([self.state, left, down, right, up]), {}
    
    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.height -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.width -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        self.state = self._coordinateToState(x, y)
        
        result = self.map[y][x]
        done = result in ['H', 'G']
        if result == 'G' :
            reward = 1
        elif result == 'H' :
            reward = -1
        else :
            reward = -1/(self.width * self.height)

        left, down, right, up = self._getNeighbors(self.state)
        return np.array([self.state, left, down, right, up]), reward, done, False, {}
    
    
    def _stateToCoordinate(self, state) :
        x = state % self.width
        y = state // self.width
        return x, y
    
    def _coordinateToState(self, x, y) :
        return self.width*y + x

    def _getNeighbors(self, state) :
        x, y = self._stateToCoordinate(state)
        if x == 0 :  # Retrieve left cell
            left = self.cell_values['W']
        else :
            left = self.cell_values[self.map[y][x-1]]

        if y == self.height -1 :  # Retrieve down cell
            down = self.cell_values["W"]
        else :
            down = self.cell_values[self.map[y+1][x]]

        if x == self.width -1 :   # Retrieve right cell
            right = self.cell_values["W"]
        else :
            right = self.cell_values[self.map[y][x+1]]

        if y == 0 :   # Retrieve up cell
            up = self.cell_values["W"]
        else :
            up = self.cell_values[self.map[y-1][x]]

        return left, down, right, up


    
    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.height) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.width) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)

### Labyrinthe 4

Même distribution de récompense et même observations que labyrinthe 3.

Mais cette fois-ci la position n'est pas codée en faisant usgae du one hot encoding.

In [ ]:
class FrozenLake4Env(gymnasium.Env) :
    def __init__(self, options = None) :
        super(FrozenLake4Env, self).__init__()
        self.map = options['lake_map']
        self.width = len(self.map[0])
        self.height = len(self.map)
        self.cell_values = {'F' : 0, 'S' : 0, 'H' : 1, 'G' : 2, 'W' : 3}
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Dict({
            'row_position' : spaces.Box(low = 0, high = self.width - 1, dtype = int),
            'col_position' : spaces.Box(low = 0, high = self.height - 1, dtype = int),
            'left_neighbor' : spaces.Discrete(4),
            'down_neighbor' : spaces.Discrete(4),
            'right_neighbor' : spaces.Discrete(4),
            'up_neighbor' : spaces.Discrete(4)
        })
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    break
        

    
    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0

        for x in range(self.width) :
            for y in range(self.height) :
                if self.map[y][x] == 'S' :
                    self.state = self._coordinateToState(x, y)
                    self.x = x
                    self.y = y
                    break


        left, down, right, up = self._getNeighbors(self.state)
        toReturn = {
            'row_position' : np.array([self.x]),
            'col_position' : np.array([self.y]),
            'left_neighbor' : left,
            'down_neighbor' : down,
            'right_neighbor' : right,
            'up_neighbor' : up
        }
        return toReturn, {}
    
    def step(self, action) :
        self.x, self.y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if self.x > 0 :
                self.x -= 1
        elif action == 1 : # Down
            if self.y < self.height -1 :
                self.y += 1
        elif action == 2 : # Right
            if self.x < self.width -1 :
                self.x += 1
        elif action == 3 : # Up
            if self.y > 0 :
                self.y -= 1
        
        self.state = self._coordinateToState(self.x, self.y)
        
        result = self.map[self.y][self.x]
        done = result in ['H', 'G']
        if result == 'G' :
            reward = 1
        elif result == 'H' :
            reward = -1
        else :
            reward = -1/(self.width * self.height)

        left, down, right, up = self._getNeighbors(self.state)
        toReturn = {
            'row_position' : np.array([self.x]),
            'col_position' : np.array([self.y]),
            'left_neighbor' : left,
            'down_neighbor' : down,
            'right_neighbor' : right,
            'up_neighbor' : up
        }
        return toReturn, reward, done, False, {}
    
    
    def _stateToCoordinate(self, state) :
        x = state % self.width
        y = state // self.width
        return x, y
    
    def _coordinateToState(self, x, y) :
        return self.width*y + x

    def _getNeighbors(self, state) :
        x, y = self._stateToCoordinate(state)
        if x == 0 :  # Retrieve left cell
            left = self.cell_values['W']
        else :
            left = self.cell_values[self.map[y][x-1]]

        if y == self.height -1 :  # Retrieve down cell
            down = self.cell_values["W"]
        else :
            down = self.cell_values[self.map[y+1][x]]

        if x == self.width -1 :   # Retrieve right cell
            right = self.cell_values["W"]
        else :
            right = self.cell_values[self.map[y][x+1]]

        if y == 0 :   # Retrieve up cell
            up = self.cell_values["W"]
        else :
            up = self.cell_values[self.map[y-1][x]]

        return left, down, right, up


    
    def render(self, mode = 'human') :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.height) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.width) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)

## Livraison

Exactement le même problème que le labyrinthe sauf qu'il faut récupérer une boîte dans le labyrinthe avant d'atteinde la sortie

### Livraison 1

Observation : Un seul entier décrivant à la fois la position de l'agent et s'il possède la boîte.

Récompenses :
- Tomber dans un trou : -1
- Sortir avec la boîte : 1
- Sinon : -1/nb_cases

In [ ]:
class LivraisonEnv(gymnasium.Env) :
    def __init__(self, options = None) :
        super(LivraisonEnv, self).__init__()
        self.map = options['lake_map']
        self.size = len(self.map)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Discrete(2 * (self.size ** 2))
        self.state = 0
        self.hasBox = False

    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0
        self.hasBox = False
        return np.array(self.state), {}

    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.size -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.size -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        
        result = self.map[y][x]
        
        if result == 'B' :
            self.hasBox = True

        self.state = self._coordinateToState(x, y)

        
        if result == 'G' and self.hasBox :
            reward = 1
            done = True
        elif result == 'H' :
            reward = -1
            done = True
        else :
            reward = -1/(self.size**2)
            done = False
        return np.array(self.state), reward, done, False, {}

    
    def _stateToCoordinate(self, state) :
        if state >= self.size ** 2 :
            state -= self.size ** 2
        x = state % self.size
        y = state // self.size
        return x, y

    
    def _coordinateToState(self, x, y) :
        toReturn = self.size*y + x
        if self.hasBox :
            toReturn += self.size ** 2
        return toReturn

    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.size) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.size) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)
        if self.state == (self.size**2) - 1 :
            print("Objectif atteint")

### Livraison 2

Observation : Position de l'agent et possession de la boîte (séparément)

Récompenses :
- Tomber dans un trou : -1
- Sortir avec la boîte : 1
- Sinon : -1/nb_cases

In [ ]:
class LivraisonRLlibEnv(gymnasium.Env) :
    def __init__(self, options = None) :
        super(LivraisonRLlibEnv, self).__init__()
        self.map = options['carte']
        self.size = len(self.map)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.MultiDiscrete([self.size ** 2, 2])
        self.state = 0
        self.hasBox = False

    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0
        self.hasBox = False
        return np.array([self.state, 0]), {}

    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.size -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.size -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        
        result = self.map[y][x]
        
        if result == 'B' :
            self.hasBox = True

        self.state = self._coordinateToState(x, y)

        
        if result == 'G' and self.hasBox :
            reward = 1
            done = True
        elif result == 'H' :
            reward = -1
            done = True
        else :
            reward = -1/(self.size**2)
            done = False

        if self.hasBox :
            return np.array([self.state, 1]), reward, done, False, {}
        else :
            return np.array([self.state, 0]), reward, done, False, {}

    
    def _stateToCoordinate(self, state) :
        x = state % self.size
        y = state // self.size
        return x, y

    
    def _coordinateToState(self, x, y) :
        toReturn = self.size*y + x
        return toReturn

    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.size) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.size) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)
        if self.state == (self.size**2) - 1 :
            print("Objectif atteint")

### Livraison 3

Coordonnées et possession codées différement : codage en un seul neurone pour coordonnées et possession
+ observe les cases adjacentes.


Pour DQN, nous aurons 3 neurones pour coder la position et la possession de la boîte.
Deux pour les coordonnées qui fournissent les entiers correspondants à la position de l'agent
Et un qui renvoie 0 si l'agent possède la boîte, 1 sinon.

In [ ]:
class Livraison3Env(gymnasium.Env) :
    def __init__(self, options = None) :
        super(Livraison3Env, self).__init__()
        self.cell_values = {'F' : 0, 'S' : 0, 'H' : 1, 'G' : 2, 'B' : 3, 'W' : 4}

        self.map = options['carte']
        
        self.size = len(self.map)
        self.width = len(self.map[0])
        self.height = len(self.map)
        
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Dict({
            'row_position' : spaces.Box(low = 0, high = self.width - 1, dtype = int),
            'col_position' : spaces.Box(low = 0, high = self.height - 1, dtype = int),
            'possession' : spaces.Box(low = 0, high = 1, dtype = int),
            'left_neighbor' : spaces.Discrete(5),
            'down_neighbor' : spaces.Discrete(5),
            'right_neighbor' : spaces.Discrete(5),
            'up_neighbor' :spaces.Discrete(5)
        })
        self.state = 0
        self.x = 0
        self.y = 0
        self.hasBox = False

    def reset(self, seed = None, options = None) :
        super().reset(seed = seed)
        self.state = 0
        self.hasBox = False
        left, down, right, up = self._getNeighbors(self.state)
        toReturn = {
            "row_position" : np.array([0]),
            "col_position" : np.array([0]),
            "possession" : np.array([0 if self.hasBox else 1]),
            'left_neighbor' : left,
            'down_neighbor' : down,
            'right_neighbor' : right,
            'up_neighbor' : up
        }
        return toReturn, {}

    def step(self, action) :
        x, y = self._stateToCoordinate(self.state)
        if action == 0 : # Left
            if x > 0 :
                x -= 1
        elif action == 1 : # Down
            if y < self.size -1 :
                y += 1
        elif action == 2 : # Right
            if x < self.size -1 :
                x += 1
        elif action == 3 : # Up
            if y > 0 :
                y -= 1
        
        
        result = self.map[y][x]
        
        if result == 'B' :
            self.hasBox = True

        self.state = self._coordinateToState(x, y)

        
        if result == 'G' and self.hasBox :
            reward = 1
            done = True
        elif result == 'H' :
            reward = -1
            done = True
        else :
            reward = -1/(self.size**2)
            done = False

        left, down, right, up = self._getNeighbors(self.state)

        toReturn = {
            "row_position" : [0],
            "col_position" : [0],
            "possession" : [0 if self.hasBox else 1],
            'left_neighbor' : left,
            'down_neighbor' : down,
            'right_neighbor' : right,
            'up_neighbor' : up
        }

        return toReturn, reward, done, False, {}


    
    def _stateToCoordinate(self, state) :
        x = state % self.size
        y = state // self.size
        return x, y

    
    def _coordinateToState(self, x, y) :
        toReturn = self.size*y + x
        return toReturn


    def _getNeighbors(self, state) :
        x, y = self._stateToCoordinate(state)
        if x == 0 :  # Retrieve left cell
            left = self.cell_values['W']
        else :
            left = self.cell_values[self.map[y][x-1]]

        if y == self.height -1 :  # Retrieve down cell
            down = self.cell_values["W"]
        else :
            down = self.cell_values[self.map[y+1][x]]

        if x == self.width -1 :   # Retrieve right cell
            right = self.cell_values["W"]
        else :
            right = self.cell_values[self.map[y][x+1]]

        if y == 0 :   # Retrieve up cell
            up = self.cell_values["W"]
        else :
            up = self.cell_values[self.map[y-1][x]]

        return left, down, right, up
    

    
    def render(self) :
        x, y = self._stateToCoordinate(self.state)
        toPrint = ''
        for i in range(self.size) :
            toPrintRow = ''
            row = self.map[i]
            for j in range(self.size) :
                if (j, i) == (x, y):
                    toPrintRow += '.'
                else :
                    toPrintRow += row[j]
            toPrint += toPrintRow + '\n'
        print(toPrint)
        if self.state == (self.size**2) - 1 :
            print("Objectif atteint")

## RomeAlésia

Jeu qui oppose 2 joueurs.

L'un joue Rome, l'autre Alésia.

Le but : capturer la ville adverse.

Début de partie :
- le même nombre d'unité est attribué aux deux joueurs.
- le front commence en milieu de plateau

Voici un exemple de plateau


R--|--|--X--|--|--A


Avec :
- R = Rome
- X = Front
- A = Alésia
- | = Étape


Si le front atteint la ville adverse vous gagnez. (Si X atteint R, Alésia gagne).


À chaque tour les deux joueurs jouent simultanément un nombre d'unité.

Ce nombre est déduit de leur total d'unité respectif.


Le joueur ayant placé le plus d'unité déplace le front d'une étape vers la ville adverse.



Fins possibles :
- Le front atteint Rome : Alésia gagne
- Le front atteint Alésia : Rome gagne
- Les deux joueurs n'ont plus d'unité alors que le front n'a atteint aucune ville : Égalité.

Plusieurs versions de ce problème sont implémentées selon la stratégie utilisée par l'adversaire.